# SmartResume — Colab / Jupyter Notebook

Self-contained notebook that mirrors the Windows `setup.bat` + `run_processor.bat` flow for use in Google Colab or any Jupyter server (Linux / macOS / Windows).

**Pipeline replicated from `process_resumes_windows.py`:**
1. Environment detection (Colab vs local Jupyter, GPU availability)
2. Repo bootstrap (clone if missing)
3. Install PyTorch (CUDA wheels on GPU, CPU otherwise) + remaining deps
4. `pip install -e .` to register the `smartresume` package
5. Bootstrap `configs/config.yaml` (copy from example if absent)
6. Download Qwen3-0.6B (LLM) + YOLOv10 (layout) from ModelScope
7. Run `ResumeAnalyzer.pipeline()` on a single PDF or a directory
8. Flatten results to a CSV (same schema as the Windows batch processor)

Run cells top-to-bottom. The first time on Colab takes ~10–15 min (model downloads ~1.5 GB).

## 1. Environment detection

In [ ]:
import os, sys, platform, subprocess, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
PY_VER = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"

print(f"Platform:       {platform.system()} {platform.release()}")
print(f"Python:         {PY_VER}  ({sys.executable})")
print(f"In Colab:       {IN_COLAB}")

# Detect NVIDIA GPU via nvidia-smi (works before torch is installed)
HAS_GPU = shutil.which('nvidia-smi') is not None and \
          subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
print(f"NVIDIA GPU:     {HAS_GPU}")
if HAS_GPU:
    out = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
                          '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
    print(f"GPU info:       {out}")

## 2. Repo bootstrap

On Colab we clone the upstream repo. On a local Jupyter server we assume this notebook lives inside the repo (e.g. `notebooks/SmartResume_Colab.ipynb`) and walk up to find it.

In [ ]:
REPO_URL = "https://github.com/alibaba/SmartResume.git"

def find_repo_root(start: Path) -> Path | None:
    """Walk up from `start` looking for the SmartResume project marker."""
    for p in [start, *start.parents]:
        if (p / 'smartresume' / '__init__.py').exists() and (p / 'pyproject.toml').exists():
            return p
    return None

if IN_COLAB:
    REPO_DIR = Path('/content/SmartResume')
    if not REPO_DIR.exists():
        print(f"Cloning {REPO_URL} -> {REPO_DIR}")
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
    else:
        print(f"Repo already present at {REPO_DIR}")
else:
    found = find_repo_root(Path.cwd().resolve())
    if found is None:
        raise RuntimeError(
            "Could not find SmartResume repo root. Place this notebook inside the repo "
            "(e.g. notebooks/) or set REPO_DIR manually."
        )
    REPO_DIR = found
    print(f"Repo root:      {REPO_DIR}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"CWD:            {Path.cwd()}")

## 3. Install dependencies

PyTorch is installed first from the appropriate index:
- **GPU host** → CUDA 12.1 wheels (Colab default, broadly compatible)
- **CPU only** → standard PyPI wheels

We mirror `setup.bat`'s pattern of uninstalling any pre-existing torch first so pip doesn't silently keep an incompatible build.

In [ ]:
PIP = [sys.executable, '-m', 'pip', 'install', '-q']

def pip_install(*pkgs, extra_args=()):
    cmd = PIP + list(extra_args) + list(pkgs)
    print('>>>', ' '.join(cmd))
    subprocess.check_call(cmd)

# --- PyTorch (force a clean install on the right index) ---
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision'],
               capture_output=True)

if HAS_GPU:
    pip_install('torch', 'torchvision',
                extra_args=('--index-url', 'https://download.pytorch.org/whl/cu121'))
else:
    pip_install('torch', 'torchvision')

# --- Remaining runtime deps (mirrors pyproject.toml) ---
pip_install(
    'transformers>=4.30.0', 'accelerate>=0.20.0', 'modelscope>=1.9.0',
    'pdfplumber>=0.10.0', 'easyocr>=1.7.0', 'opencv-python>=4.7',
    'pillow>=9.5', 'numpy>=1.23', 'pandas', 'tqdm', 'pyyaml>=6.0',
    'tiktoken>=0.5.1', 'openai>=1.30.0', 'json-repair>=0.16.0',
    'langdetect>=1.0.9', 'requests>=2.25.0', 'onnxruntime>=1.17.0',
)

# --- SmartResume itself, in editable mode ---
pip_install('-e', '.')
print('Dependencies installed.')

## 4. Verify install + GPU

If you just installed torch in this session, restart the runtime once before continuing (Colab: *Runtime → Restart session*). Otherwise the cached pre-install torch may still be loaded.

In [ ]:
import importlib
for mod in ['torch', 'transformers', 'pdfplumber', 'easyocr', 'pandas', 'modelscope']:
    try:
        m = importlib.import_module(mod)
        print(f"{mod:14s} {getattr(m, '__version__', 'ok')}")
    except ImportError as e:
        print(f"{mod:14s} MISSING ({e})")

import torch
print()
print(f"torch.__version__:      {torch.__version__}")
print(f"CUDA available:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device:            {torch.cuda.get_device_name(0)}")
    cap = torch.cuda.get_device_capability(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Compute capability:     {cap[0]}.{cap[1]}")
    print(f"VRAM:                   {total:.1f} GB")

## 5. Config bootstrap

`smartresume.utils.config` looks for `configs/config.yaml`. Copy from the example if missing, then enable direct (in-process) model loading so we don't need a vLLM server.

In [ ]:
import yaml

CFG_PATH    = REPO_DIR / 'configs' / 'config.yaml'
CFG_EXAMPLE = REPO_DIR / 'configs' / 'config.yaml.example'

if not CFG_PATH.exists():
    if not CFG_EXAMPLE.exists():
        raise FileNotFoundError(f"Neither {CFG_PATH} nor {CFG_EXAMPLE} exists.")
    shutil.copy(CFG_EXAMPLE, CFG_PATH)
    print(f"Created {CFG_PATH} from example.")

with open(CFG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f) or {}

# Force direct (no API server) mode — same as the Windows path
cfg['use_direct_models']  = True
cfg['direct_model_name']  = 'models/Qwen3-0.6B'
cfg.setdefault('processing', {})
cfg.setdefault('layout_detection', {})['enabled'] = True
cfg.setdefault('ocr', {})['use_cuda'] = bool(torch.cuda.is_available())

with open(CFG_PATH, 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print(f"Config written to {CFG_PATH}")
print(f"  use_direct_models: {cfg['use_direct_models']}")
print(f"  direct_model_name: {cfg['direct_model_name']}")
print(f"  ocr.use_cuda:      {cfg['ocr']['use_cuda']}")

## 6. Download models from ModelScope

Pulls Qwen3-0.6B (LLM, ~1.2 GB) and YOLOv10 (layout, ~30 MB) into `models/`. Skipped automatically if already present.

In [ ]:
MODELS_DIR = REPO_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)

qwen_done   = (MODELS_DIR / 'Qwen3-0.6B' / 'config.json').exists()
layout_done = (MODELS_DIR / 'yolov10' / 'best.onnx').exists()

if qwen_done and layout_done:
    print('Both models already downloaded — skipping.')
else:
    from smartresume.utils.models_download_utils import download_model
    from smartresume.utils.model_paths import ModelType, ModelSource

    if not qwen_done:
        print('Downloading Qwen3-0.6B (LLM) ...')
        path = download_model(ModelType.LLM, ModelSource.MODELSCOPE, str(MODELS_DIR))
        print(f'  -> {path}')

    if not layout_done:
        print('Downloading YOLOv10 (layout) ...')
        path = download_model(ModelType.LAYOUT, ModelSource.MODELSCOPE, str(MODELS_DIR))
        print(f'  -> {path}')

for p in sorted(MODELS_DIR.rglob('*'))[:30]:
    if p.is_file():
        print(f'  {p.relative_to(MODELS_DIR)}  ({p.stat().st_size/1024**2:.1f} MB)')

## 7. Initialize the analyzer

First call loads Qwen3-0.6B + EasyOCR + the layout detector — about 30–60 s on GPU, several minutes on CPU.

In [ ]:
# Force-reload config so the YAML edits above take effect
import importlib, smartresume.utils.config as _cfg
importlib.reload(_cfg)

from smartresume import ResumeAnalyzer

analyzer = ResumeAnalyzer(init_ocr=True, init_llm=True)
print('ResumeAnalyzer ready.')

## 8. Process a single resume

Set `RESUME_PATH` to your PDF. On Colab you can upload via the file panel or `files.upload()`.

In [ ]:
# --- Pick a resume ---
RESUME_PATH: Path | None = None

if IN_COLAB and RESUME_PATH is None:
    from google.colab import files
    print('Upload a PDF resume:')
    uploaded = files.upload()
    if uploaded:
        RESUME_PATH = Path(next(iter(uploaded)))

if RESUME_PATH is None:
    # Local fallback: first PDF in resumes/
    candidates = sorted((REPO_DIR / 'resumes').glob('*.pdf')) if (REPO_DIR / 'resumes').exists() else []
    if candidates:
        RESUME_PATH = candidates[0]

if RESUME_PATH is None or not Path(RESUME_PATH).exists():
    raise FileNotFoundError('Set RESUME_PATH to a PDF before running this cell.')

print(f'Processing: {RESUME_PATH}')
result = analyzer.pipeline(
    cv_path=str(RESUME_PATH),
    resume_id=Path(RESUME_PATH).stem,
    extract_types=['basic_info', 'work_experience', 'education', 'gpa'],
)

import json
print(json.dumps(result, ensure_ascii=False, indent=2)[:4000])

## 9. Batch a directory → CSV

Same schema as `process_resumes_windows.py` (`CSV_COLUMNS` / `flatten_extracted_data`).

In [ ]:
import time, pandas as pd

RESUMES_DIR = REPO_DIR / 'resumes'
OUTPUT_CSV  = REPO_DIR / 'resume_results.csv'

CSV_COLUMNS = [
    'filename', 'extraction_status', 'full_name', 'email', 'phone', 'location',
    'work_companies', 'positions', 'education_schools', 'education_degrees',
    'gpa', 'honors', 'error_message',
]

def flatten(raw: dict) -> dict:
    basic = raw.get('basicInfo') or {}
    work  = raw.get('workExperience') or []
    edu   = raw.get('education') or []
    gpa   = raw.get('gpa') or {}
    return {
        'full_name':         basic.get('name', ''),
        'email':             basic.get('personalEmail', ''),
        'phone':             basic.get('phoneNumber', ''),
        'location':          basic.get('currentLocation', ''),
        'work_companies':    '; '.join(w.get('companyName', '') for w in work if w.get('companyName')),
        'positions':         '; '.join(w.get('position', '')    for w in work if w.get('position')),
        'education_schools': '; '.join(e.get('school', '')      for e in edu  if e.get('school')),
        'education_degrees': '; '.join(e.get('degreeLevel', '') for e in edu  if e.get('degreeLevel')),
        'gpa':               gpa.get('value', ''),
        'honors':            gpa.get('honors', ''),
    }

if not RESUMES_DIR.exists():
    print(f'No directory {RESUMES_DIR} — skipping batch step.')
else:
    pdfs = sorted(p for p in RESUMES_DIR.iterdir() if p.suffix.lower() == '.pdf')
    print(f'Found {len(pdfs)} PDF(s) in {RESUMES_DIR}')

    rows = []
    t0 = time.time()
    for i, pdf in enumerate(pdfs, 1):
        ts = time.time()
        row = {c: '' for c in CSV_COLUMNS}
        row['filename'] = pdf.name
        try:
            raw = analyzer.pipeline(
                cv_path=str(pdf),
                resume_id=f'resume_{i:04d}',
                extract_types=['basic_info', 'work_experience', 'education', 'gpa'],
            )
            if raw and 'error' not in raw:
                row.update(flatten(raw))
                row['extraction_status'] = 'success'
            else:
                row['extraction_status'] = 'empty' if not raw else 'error'
                row['error_message']     = (raw or {}).get('error', '')
        except Exception as e:
            row['extraction_status'] = 'error'
            row['error_message']     = str(e)
        rows.append(row)
        print(f'  [{i}/{len(pdfs)}] {pdf.name}  ({time.time()-ts:.1f}s)  -> {row["extraction_status"]}')

    df = pd.DataFrame(rows, columns=CSV_COLUMNS)
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f'\nSaved {len(df)} rows -> {OUTPUT_CSV}  ({time.time()-t0:.1f}s total)')
    df.head(10)

## 10. (Optional) Download the CSV

On Colab this triggers a browser download. Locally just open the CSV from disk.

In [ ]:
if IN_COLAB and OUTPUT_CSV.exists():
    from google.colab import files
    files.download(str(OUTPUT_CSV))